# Inflation Fell; Prices Did Not — Reproducible walk-through

Article 26, United States 2019–2025. Frozen official government observations, 2026 retrospective data vintage. Seven figures are descriptive and the eighth is an invented selection example. None identifies causal policy effects.

In [ ]:
from prepare_data import ROOT, monthly_rows, build_panel
from analysis import FIGURE_NAMES, endpoint_comparisons, recovery_milestones, wage_selection_mean, make_figures
monthly = monthly_rows()
annual = build_panel()
print('Observed calendar slots:', len(monthly), 'Annual rows:', len(annual))

## The October 2025 data gap

The federal funding lapse prevented collection of October 2025 household survey and most CPI estimates. Missing values are not zeros and are never interpolated. The monthly wage series has a value, but its real purchasing power cannot be calculated without an observed CPI.

In [ ]:
oct25 = next(r for r in monthly if r['observation_date'] == '2025-10-01')
assert oct25['total_unemployment_pct'] is None
assert oct25['cpi_sa_index_dec2019'] is None
assert oct25['real_earnings_index_dec2019'] is None
assert oct25['nominal_hourly_earnings_usd'] is not None
dec25 = next(r for r in monthly if r['observation_date'] == '2025-12-01')
print('October missing CPS / CPI:', oct25['total_unemployment_pct'], oct25['cpi_sa_index_dec2019'])
print('December 2025 accumulated CPI increase:', round(float(dec25['cpi_sa_index_dec2019']) - 100, 2), '%')
print('December 2025 annual CPI inflation:', round(float(dec25['cpi_nsa_12m_inflation_pct']), 2), '%')

## Different recovery benchmarks

Real GDP is compared with 2019Q4, payroll jobs with February 2020, and real median household income with annual 2019. These are crossings of observed old levels, not estimates of a no-pandemic growth path.

In [ ]:
for row in recovery_milestones():
    print(row['indicator'], 'first recovered', row['first_at_or_above_date'])
for row in endpoint_comparisons():
    print(row['indicator'], round(float(row['percent_change']), 2), '% change')

## Hypothetical wage selection

A 50/50 workforce at hourly pay of $15 and $35 has no individual raises. Layoffs concentrated among the lower-paid workers still raise the observed average wage among remaining employees. This is entirely invented arithmetic, never a calibration to the pandemic.

In [ ]:
start = wage_selection_mean(15.0, 35.0, 0.0)
half_low_jobs_lost = wage_selection_mean(15.0, 35.0, 0.5)
assert abs(start - 25.0) < 1e-10
assert abs(half_low_jobs_lost - 28.33333333333) < 1e-8
print('Fixed individual wages; observed average:', start, 'to', round(half_low_jobs_lost, 2))

## Reproduce and audit eight graphics

Regenerate original images using only the archived data; then verify every SHA-256 digest in the frozen source manifest.

In [ ]:
import hashlib
import json
make_figures(ROOT)
manifest = json.loads((ROOT / 'data' / 'sources.json').read_text(encoding='utf-8'))
for entry in manifest['sources']:
    assert hashlib.sha256((ROOT / entry['file']).read_bytes()).hexdigest() == entry['sha256']
assert len(FIGURE_NAMES) == 8
assert all((ROOT / 'figures' / name).is_file() for name in FIGURE_NAMES)
print('Seven source hashes verified; eight figures regenerated.')

## Visual result: inflation rate versus accumulated price level

![Accumulated consumer prices versus annual inflation](figures/figure_03_inflation_vs_price_level.png)

The rate fell substantially after mid-2022; the index level stayed much higher than before the pandemic.